<a href="https://colab.research.google.com/github/Jasmin08Coder93ML/ML_Prediction_Project-/blob/main/Customer_Churn_Prediction_Project02_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Устанавливаем библиотеку для работы с Telegram Bot API
!pip install pyTelegramBotAPI --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.3/48.3 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.4/308.4 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.6/216.6 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 10.3 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: idna
    Found existing installation: idna 3.13
    Uninstalling idna-3.13:
      Successfully uninstalled idna-3.13
  Attempting uninstall: charset_normalizer
    Found existing ins

In [ ]:
# To find your Telegram ID, you can send any message to the bot and check `message.chat.id`.
# For example, after running the bot, send `/start` and your ID will be printed in the console if you add a print statement inside `welcome` for `message.chat.id`.
MY_TELEGRAM_ID = 6427400843  # <-- REPLACE THIS WITH YOUR ACTUAL TELEGRAM ID

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import joblib
from google.colab import files

# 1. Создаем искусственные данные (чтобы не искать файлы)
data = {
    'стаж_месяцев': np.random.randint(1, 72, 1000),
    'ежемесячный_платеж': np.random.randint(20, 120, 1000),
    'тех_поддержка': np.random.randint(0, 2, 1000), # 0 - нет, 1 - есть
    'ушел': np.random.randint(0, 2, 1000) # Цель: 0 - остался, 1 - ушел
}
df = pd.DataFrame(data)

# 2. Выделяем данные для обучения (X) и ответ (y)
X = df[['стаж_месяцев', 'ежемесячный_платеж', 'тех_поддержка']]
y = df['ушел']

# 3. Делим данные на обучающие и тестовые
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Создаем и обучаем модель (Логистическая регрессия)
model = LogisticRegression()
model.fit(X_train, y_train)

# 5. Проверяем точность
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"Поздравляю! Модель обучена.")
print(f"Точность твоего первого алгоритма: {accuracy:.2%}")

# Сохраняем модель
joblib.dump(model, 'churn_model.pkl')

# Скачиваем готовый файл
# files.download('churn_model.pkl') # Закомментирована строка, которая удаляет файл из Colab

Поздравляю! Модель обучена.
Точность твоего первого алгоритма: 46.50%


['churn_model.pkl']



В этом проекте я использую логистическую регрессию для прогнозирования оттока клиентов. Это помогает бизнесу заранее узнавать, кто из пользователей может перестать пользоваться услугами.

In [ ]:
import telebot
import pandas as pd
import numpy as np
import joblib
import threading
from telebot import types
from sklearn.linear_model import LogisticRegression
from datetime import datetime

# ========================================================
# 1. ПОДГОТОВКА МОДЕЛИ (для первого бота)
# ========================================================
data = {
    'стаж_месяцев': np.random.randint(1, 72, 500),
    'ежемесячный_платеж': np.random.randint(20, 150, 500),
    'тех_поддержка': np.random.randint(0, 2, 500),
    'ушел': np.random.randint(0, 2, 500)
}
df_ml = pd.DataFrame(data)
model_ml = LogisticRegression()
model_ml.fit(df_ml[['стаж_месяцев', 'ежемесячный_платеж', 'тех_поддержка']], df_ml['ушел'])
joblib.dump(model_ml, 'churn_model.pkl')

# ========================================================
# 2. НАСТРОЙКА БОТОВ (Вставьте ваши токены!)
# ========================================================
TOKEN_ANALYZER = 'ВАШ_ТОКЕН_АНАЛИЗАТОРА'
TOKEN_ACADEMY = 'ВАШ_ТОКЕН_АКАДЕМИИ'

bot_analyst = telebot.TeleBot(TOKEN_ANALYZER)
bot_academy = telebot.TeleBot(TOKEN_ACADEMY)

# Общие переменные
user_states = {}
stats = {"total": 0, "risks": 0}

# ========================================================
# 3. ЛОГИКА БОТА-АНАЛИЗАТОРА (Jasmin Loyalty)
# ========================================================
@bot_analyst.message_handler(commands=['start'])
def start_analyst(message):
    markup = types.ReplyKeyboardMarkup(resize_keyboard=True)
    markup.add("🚀 Сделать прогноз", "📊 Статистика")
    bot_analyst.send_message(message.chat.id, "🤖 Привет! Я анализатор оттока.", reply_markup=markup)

@bot_analyst.message_handler(func=lambda m: m.text == "📊 Статистика")
def show_stats(message):
    bot_analyst.send_message(message.chat.id, f"📈 Всего прогнозов: {stats['total']}\nВыявлено рисков: {stats['risks']}")

@bot_analyst.message_handler(func=lambda m: m.text == "🚀 Сделать прогноз")
def step1(message):
    msg = bot_analyst.send_message(message.chat.id, "1️⃣ Введите стаж (мес):")
    bot_analyst.register_next_step_handler(msg, step2)

def step2(message):
    user_states[message.chat.id] = {'tenure': int(message.text)}
    msg = bot_analyst.send_message(message.chat.id, "2️⃣ Введите платеж ($):")
    bot_analyst.register_next_step_handler(msg, step3)

def step3(message):
    user_states[message.chat.id]['payment'] = float(message.text)
    msg = bot_analyst.send_message(message.chat.id, "3️⃣ Техподдержка (1-да, 0-нет):")
    bot_analyst.register_next_step_handler(msg, final_analyst)

def final_analyst(message):
    try:
        data = user_states[message.chat.id]
        input_df = pd.DataFrame([[data['tenure'], data['payment'], int(message.text)]],
                                columns=['стаж_месяцев', 'ежемесячный_платеж', 'тех_поддержка'])
        prob = joblib.load('churn_model.pkl').predict_proba(input_df)[0][1] * 100
        stats["total"] += 1
        res = "⚠️ Риск оттока!" if prob > 50 else "✅ Клиент лоялен."
        if prob > 50: stats["risks"] += 1
        bot_analyst.send_message(message.chat.id, f"{res}\nВероятность: {prob:.1f}%")
    except: bot_analyst.send_message(message.chat.id, "Ошибка в данных.")

# ========================================================
# 4. ЛОГИКА БОТА-АКАДЕМИИ (Jasmin Edu Academy)
# ========================================================
courses = {
    "it": "IT technology & QA", "dev": "Java/JS/Fullstack",
    "ai": "Machine learning & AI", "sec": "Cybersecurity & DevOps"
}

@bot_academy.message_handler(commands=['start'])
def start_academy(message):
    markup = types.ReplyKeyboardMarkup(resize_keyboard=True)
    markup.add("🎓 Подготовка к ВУЗам", "💻 IT Курсы", "🔥 СКИДКА")
    bot_academy.send_message(message.chat.id, "🌟 Добро пожаловать в Jasmine IT Academy!", reply_markup=markup)

@bot_academy.message_handler(func=lambda m: m.text == "💻 IT Курсы")
def show_courses(message):
    markup = types.InlineKeyboardMarkup()
    for k, v in courses.items():
        markup.add(types.InlineKeyboardButton(v, callback_data=f"c_{k}"))
    bot_academy.send_message(message.chat.id, "Выберите курс:", reply_markup=markup)

@bot_academy.message_handler(func=lambda m: m.text == "🔥 СКИДКА")
def discount(message):
    text = "🎁 СКИДКА! Осталось 5 мест. Старт 1 июня!\nТоропитесь записаться!"
    bot_academy.send_message(message.chat.id, text)

# ========================================================
# 5. ЗАПУСК ОБОИХ БОТОВ ОДНОВРЕМЕННО
# ========================================================
def run_analyst():
    print("Бот-анализатор запущен...")
    bot_analyst.polling(none_stop=True, interval=0)

def run_academy():
    print("Бот-академия запущен...")
    bot_academy.polling(none_stop=True, interval=0)

if __name__ == "__main__":
    # Создаем два потока
    t1 = threading.Thread(target=run_analyst)
    t2 = threading.Thread(target=run_academy)

    # Запускаем потоки
    t1.start()
    t2.start()